## **Base model and Dataset for RLHF**

In [10]:
import torch
import transformers
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM
import pandas as pd
import numpy

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.10.0+cu128
Cuda version: 12.8
transformers version: 5.3.0
GPU 사용 가능여부: True


In [2]:
from transformers import PreTrainedTokenizerFast

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/kogpt2-base-v2"

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,
    bos_token='</s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>',
    padding_side="right",
    model_max_length=512
)

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **Supervised Fine-Tuning**

In [13]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy
import json

In [12]:
# Train dataset과 eval dataset을 나눌 수 있도록 수정함

import copy
import json
import logging
import random
from typing import Sequence, Dict, List

import torch
from torch.utils.data import Dataset
import transformers


class SFTDataset(Dataset):
    def __init__(
        self,
        data_path: str,
        tokenizer: transformers.PreTrainedTokenizer,
        split: str = "train",          # "train" or "eval"
        eval_ratio: float = 0.1,       # eval 비율
        seed: int = 42,                # split 고정용
        verbose: bool = False
    ):
        super().__init__()
        logging.warning(f"Loading data... split={split}")

        if split not in ["train", "eval"]:
            raise ValueError("split must be either 'train' or 'eval'")

        if not (0.0 < eval_ratio < 1.0):
            raise ValueError("eval_ratio must be between 0 and 1")

        pattern_instruction = "prompt"
        pattern_output = "completion"

        with open(data_path, "r", encoding="utf-8-sig") as json_file:
            list_data_dict = json.load(json_file)

        # -----------------------------
        # 1. train / eval split
        # -----------------------------
        total_size = len(list_data_dict)
        indices = list(range(total_size))

        rng = random.Random(seed)
        rng.shuffle(indices)

        eval_size = int(total_size * eval_ratio)
        eval_indices = set(indices[:eval_size])
        train_indices = set(indices[eval_size:])

        if split == "train":
            selected_data = [list_data_dict[i] for i in range(total_size) if i in train_indices]
        else:
            selected_data = [list_data_dict[i] for i in range(total_size) if i in eval_indices]

        if verbose:
            print(f"Total samples: {total_size}")
            print(f"Train samples: {len(train_indices)}")
            print(f"Eval samples: {len(eval_indices)}")
            print(f"Current split ({split}) samples: {len(selected_data)}")

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        sources = []
        for example in selected_data:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        targets = []
        for example in selected_data:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")

        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)
        examples_tokenized = self._tokenize_fn(examples, tokenizer)

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)

        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        self.input_ids = input_ids
        self.labels = labels

        logging.warning(f"Loading data done!! split={split}, size={len(self.labels)}")

    def _tokenize_fn(
        self,
        strings: Sequence[str],
        tokenizer: transformers.PreTrainedTokenizer
    ) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]

        input_ids = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item()
            for tokenized in tokenized_list
        ]

        return dict(
            input_ids=input_ids,
            labels=input_ids,
            input_ids_lens=input_ids_lens,
            labels_lens=input_ids_lens,
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[i],
            "labels": self.labels[i]
        }

In [14]:
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [6]:
# SFT_dataset 클래스를 사용해 훈련셋을 만들고 data collator 인스턴스를 만들기
train_dataset = SFTDataset(
    data_path='../KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl',
    tokenizer=tokenizer,
    split="train",
    eval_ratio=0.1,
    seed=42,
    verbose=True
)

val_dataset = SFTDataset(
    data_path='../KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl',
    tokenizer=tokenizer,
    split="eval",
    eval_ratio=0.1,
    seed=42,
    verbose=True
)

data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

bert_score_train_dataset = train_dataset
bert_score_val_dataset = val_dataset

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

Total samples: 12000
Train samples: 10800
Eval samples: 1200
Current split (train) samples: 10800


Total samples: 12000
Train samples: 10800
Eval samples: 1200
Current split (eval) samples: 1200


input : tensor([  739,   378,   378,   378, 14659, 13394, 37091, 10651,   383, 25841,
         8006, 14914,   375,  7673, 20479,  8091, 22311,  9036, 30902, 13675,
          375,   378,   378,   378, 41951,   454,  9549, 20549,   383,  8142,
         7192, 14914,   382, 37767, 13753,  8263,  7166,   739,  8352,  7659,
         9594, 25585, 13600,  8022,  9378, 11532,  9887, 11218,  9111, 16691,
        10351, 10561,  9128, 20479,  8091,  9065,  9446,  9036, 28420, 26521,
        10163, 26367,  6958,  9030,  9882, 12317, 25882,  9209, 37194, 10351,
         9036, 12168, 10529, 15989,  9719, 15434, 10552, 11188, 13362,  9036,
        15805, 11300, 11846,  9146, 16691,  9181,  7397, 15806, 13480, 11342,
        17596,  9161, 19996,  9025, 25006, 18595,  9966, 12592, 10751, 11814,
         8711,  9046, 12450,  9117,  7377, 12521,     1])
output: tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -10

In [7]:
print("Train dataset 길이:", len(train_dataset))
print("val dataset 길이:", len(val_dataset))

Train dataset 길이: 10800
val dataset 길이: 1200


In [8]:
import transformers
from transformers import EarlyStoppingCallback

training_args = transformers.TrainingArguments(
    output_dir="test",
    num_train_epochs=10,                 # 충분히 크게 두고 early stopping에 맡기는 편이 일반적
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16=True,

    # evaluation / saving
    eval_strategy="epoch",               # 구버전에선 evaluation_strategy일 수 있음
    eval_steps=100,
    save_strategy="epoch",
    save_steps=100,
    save_total_limit=2,

    # best model / early stopping
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # 커스텀 collator 쓸 때 안전장치
    remove_unused_columns=False,
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # 또는 eval_dataset 변수명 사용
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [9]:
# trainer.train()
# model.save_pretrained('models/output_1_SFT')

In [10]:
generator = transformers.pipeline('text-generation', model='models/output_1_SFT', tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'num_beams', 'early_stopping', 'top_k', 'do_sample', 'max_new_tokens', 'repetition_penalty', 'no_repeat_ngram_size', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer t


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 AI 어시스턴트이기 때문에 정확한 답변을 드리기 어렵습니다. "불고기용 고기의 한우는 불고기용 고기로 유명합니다.", 'token': 58}?\n\n하지만, 저는 인공지능 언어모델로써 자연어 이해 및 생성 기술을 사용하고 있습니다.

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 41대 부통령직을 수행하지 않았습니다.律)律)은 조선시대의 소설인『삼국유사(三國遺事)》에 등장하는 인물입니다.律)는 조선시대 후기 소설인 『삼국유사』에서 등장하는 인물 중 하나였습니다.律은 조선시대에 쓰이지 않았으며, 《삼국사신기》에서도

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 미국 항공국(North American International Airports)의 공식 웹사이트나 검색 엔진을 활용하여 시카고를 찾으실 수 있습니다.院)은 대한민국 경기도 성남시 분당구 구미동에 위치해 있습니다.院은

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'죄송합니다. 저는 인공지능 언어모델로써 미세먼지 여부를 파악할 수 없습니다. 제가 할 수 있는 일이라면 무엇이 문제인지 알려주시면 더 나은 답변을 드릴 수 있을 것입니다.戰)戰)은 조선시대 후기 조선시대의 소설로, 병자호란 때 청나라와의 전쟁에서 패배하고 항복하는 이야기가 그려져 있습니다.


In [11]:
torch.cuda.empty_cache()

## **Reward Model**

In [12]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

In [13]:
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [14]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
# tokenizer = AutoTokenizer.from_pretrained(
#     'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
#     padding_side="right",
#     model_max_length=512,
# )
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,
    bos_token='</s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>',
    padding_side="right",
    model_max_length=512
)


with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_head.weight                          | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
with open('../KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)



    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[45])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '애플은 리사를 어떻게 처리했어', 'chosen': '애플이 누구인지 명확히 알 수 없어서, 리사가 누구인지와 어떤 상황에서 처리되었는지에 대한 추가적인 정보가 필요합니다. 따라서, 보다 정확한 답변을 제공할 수 없습니다.', 'rejected': '애플은 리사를 위해 고객 서비스 부서에서 고객 다양한 컴퓨터 관련 문제에 대해 응답하는 데 필요한 모든 지원을 제공했습니다. 사용자가 하드웨어 문제를 경험할 때, 전문가들은 필요한 수리(수리, 추가 부품 제공, 소프트웨어 업그레이드 등)을 제공해 드릴 수 있습니다. 또한, 사용자가 사용 방법 문제나 기타 문제를 경험할 때, 대화 상대로 사용자를 지원할 수 있는 전문 고객 서비스 직원들이 사용자에게 상담하고 도움을 주는 데 도움이 될 수 있는 정보를 제공합니다. 또한, 인터넷에서 제공되는 정보를 통해 문제를 해결하거나 고객 서비스 웹 사이트를 통해 자신의 문제를 진단할 수 있도록 하는 등 다양한 방법으로 리사를 처리해 왔습니다.'}


In [16]:
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

{'prompt': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?', 'chosen': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.', 'rejected': '대구 영화사옥'}


In [17]:
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [18]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
흑고래의 무게는 어느 정도야
######################################################################
## chosen ##
흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
######################################################################
## rejected ##
흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.


In [19]:
# trainer = RewardModelTrainer(
#     model=model,
#     strategy=NaiveStrategy(),
#     optim=torch.optim.Adam(model.parameters(), lr=5e-5),
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     batch_size=4,
#     max_epochs=10,
#     early_stopping_patience=3,
#     early_stopping_min_delta=0.0,
# )

# trainer.fit(use_lora=False)

In [20]:
# model.save_pretrained('models/output_2_RM')

In [21]:
torch.cuda.empty_cache()

## **Proximal Policy Optimization**

In [22]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [23]:
with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(torch.cuda.current_device())
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        model_name,
        bos_token='</s>',
        eos_token='</s>',
        unk_token='<unk>',
        pad_token='<pad>',
        mask_token='<mask>',
        padding_side="right",
        model_max_length=512
    )

    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [24]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [25]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [26]:
with open('../KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

In [27]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [28]:
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO')

Episode [1/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [2/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [3/10]:   0%|          | 0/3 [00:00<?, ?it/s]

...

Episode [9/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [10/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
def generation(input_text, model, isPrint=False):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    
    if isPrint == True:
        print()
        print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor, isPrint=True)


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'죄송하지만, 저는 인공지능 언어모델로써 이 질문에 답변해 드릴 수 없습니다. 고기는 보통 얇게 잘라 먹거나 고기, 치즈소스 등을 섞어 만든 것으로 알려드리겠습니다. 저는 "불고기용 한우에요?"라는 질문을 안해도 됩니다. "이 질문이 무슨 질문인가요?"라고 묻을 경우 더욱 정확한 답을 드리도록 노력하겠습니다., 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子 , 庚子  , 庚子"를 의미합니다. 庚子 , 庚子 , 

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨이 41대 부통령직을 수행한 년도는 2017년입니다.典三郎,庚寅), 庚寅, 庚寅, 永獻 등이 있었습니다.典)은 당시 조선 시대 중기 인물인 김영조와 송준근이 각각 대표합니다., , , , , , , , ,  등의 이름으로 잘 알려져 있습니다.典)은 김영삼 당시 대통령의 비서로서, 김대중 전 대통령이 대선 캠프 일정을 조율할 때 노무현 전 대통령이 대선 캠프를 대행했습니다.前은 당시에는 김대 전 대통령의 캠프에서 활약하며, 김대중 전 대통령이 경선 캠프 일정 조율 및 대통령 선거 캠프 활동을 했습니다.前은 이승만 전 대통령의 캠프 파견 일정 조율에 참여하여 김대중 전 대통령과 김대중 전 대통령의 캠프 일정을 조정했습니다.前은 김홍일 전 대통령과 김영삼 전 대통령의 캠프 일정을 조정하기 위해 김영태 전 대통령이 이승만 전 대통령의 대선 캠프 일정을 조정했습니다.前은 김 전 대통령과 김영삼 전 대통령이 당시 대선 캠

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어

### Response(응답):'시카고 오헤어 국제공항은 일본 치요시 도시미나루 국제공항에 위치해 있습니다.濟圖境院)은 미국

In [30]:
import torch
from bert_score import score


def compute_bertscore_from_sft_val_dataset(
    model,
    val_dataset,
    tokenizer,
    generation_fn,   # 네 generation 함수 주입
    lang="ko",
    num_samples=None,
    skip_special_tokens=True,
    verbose=True,
):
    """
    SFT용 tensor 형식 val_dataset에서
    prompt를 복원해 generation_fn을 수행한 뒤
    generated output vs reference completion 간 BERTScore를 계산한다.
    """

    model.eval()

    predictions = []
    references = []
    prompts = []

    total_len = len(val_dataset) if num_samples is None else min(len(val_dataset), num_samples)

    for i in range(total_len):
        sample = val_dataset[i]
        input_ids = sample["input_ids"]
        labels = sample["labels"]

        if not isinstance(input_ids, torch.Tensor):
            input_ids = torch.tensor(input_ids)
        if not isinstance(labels, torch.Tensor):
            labels = torch.tensor(labels)

        # 1) prompt 복원
        prompt_mask = (labels == -100)
        prompt_ids = input_ids[prompt_mask]

        if prompt_ids.numel() == 0:
            prompt_ids = input_ids

        prompt_text = tokenizer.decode(prompt_ids, skip_special_tokens=skip_special_tokens)

        # 2) reference 복원
        target_ids = labels[labels != -100]

        if tokenizer.pad_token_id is not None:
            target_ids = target_ids[target_ids != tokenizer.pad_token_id]

        reference_text = tokenizer.decode(target_ids, skip_special_tokens=skip_special_tokens)

        # 3) generation 함수 사용
        full_output_text = generation_fn(prompt_text, model)

        # generation 함수가 prompt+답변 전체를 반환한다고 가정하고,
        # prompt 이후 부분만 prediction으로 사용
        if full_output_text.startswith(prompt_text):
            prediction_text = full_output_text[len(prompt_text):].strip()
        else:
            # 혹시 정확히 prefix 매칭이 안 되면 전체 출력 사용
            prediction_text = full_output_text.strip()

        prompts.append(prompt_text)
        references.append(reference_text)
        predictions.append(prediction_text)

        if verbose and i < 2:
            print(f"\n[Sample {i}]")
            print(f"PROMPT:\n{prompt_text}")
            print(f"REFERENCE:\n{reference_text}")
            print(f"PREDICTION:\n{prediction_text}")

    # 4) BERTScore 계산
    P, R, F1 = score(predictions, references, lang=lang, verbose=verbose)

    result_dict = {
        "mean_precision": P.mean().item(),
        "mean_recall": R.mean().item(),
        "mean_f1": F1.mean().item(),
        "predictions": predictions,
        "references": references,
        "prompts": prompts,
        "P": P,
        "R": R,
        "F1": F1,
    }

    return result_dict

In [31]:
result_list = []

for i in range(3):
    print(f"TASK-{i}")
    result = compute_bertscore_from_sft_val_dataset(
        model=actor,
        val_dataset=bert_score_val_dataset,
        tokenizer=tokenizer,
        generation_fn=generation,
        lang="ko",
        num_samples=100,
        verbose=True,
    )

    result_list.append(result["mean_f1"])


# print("Mean Precision:", result["mean_precision"])
# print("Mean Recall:", result["mean_recall"])
print("Mean F1:", sum(result_list)/len(result_list))

TASK-0

[Sample 0]
PROMPT:
### Instruction(명령어):
술 먹고 싶어

### Response(응답):
REFERENCE:
'술은 알코올이 함유된 음료수이며, 건강에 나쁜 영향을 미칠 수 있습니다. 따라서 술을 마시기 전에 본인의 건강 상태와 함께 적당한 양을 마시는 것이 중요합니다. 술을 마시기 전에 반드시 20세 이상인지 확인하시고, 규정된 양 이내에서 적절하게 즐기시길 바랍니다. 또한 술을 마시고 운전하는 것은 절대 금지하시기 바랍니다.
PREDICTION:
'저는 술을 마시지 못합니다. 하지만 술을 마시면 건강상에 좋지 않을 수 있기 때문에 적극적으로 추천드립니다.\n\n또한, 술을 마신다 해도 몸에 유해한 물질이 많이 들어간다면 의사와 상담하여 원인을 파악해야 합니다. 유해한 물질은 체내 체내 노폐물과 기분 전환에 도움을 줄 수 있으므로, 건강 유지를 위해 충분한 휴식과 함께 적절한 맥주 섭취는 권장합니다.\n\n또한, 술 마신다고 해서 반드시 취하지 말고, 술에 취하지 말고, 적극적으로 마시면 혈당 수준을 유지하며 건강 유지에 도움이 됩니다.\n\n마지막으로, 술을 마신지 않는 것이 오히려 건강에도 좋지 않을 수 있습니다. 음주는 음주자중일 때에는 자제하고, 다른 사람들과 함께 시간을 즐기기 위해 노력해야 합니다.\n\n마지막으로, 술 마신다면 주변 환경을 충분히 고려하는 것이 중요합니다. 술을 마시는 것은 건강에 문제가 될 수 있으므로 자제하세요. \n\n마지막으로, 술 마시는 것이 건강에 해롭다는 것은 중요합니다. 건강한 생활습관을 유지하는 것은 뇌 건강을 유지하는데 도움이 됩니다. 술 마시는 대신, 술 마시기, 취미생활 등을 하며 건강에도 유의하

[Sample 1]
PROMPT:
### Instruction(명령어):
쇼와 천황의 면책에 불만을 표한 프랑스의 판사는?

### Response(응답):
REFERENCE:
'제 2차 세계 대전 전후의 재앙적인 상황에서 쇼-고라제 정부와 천황을 비난하는 것은 위험하

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 0.70 seconds, 142.15 sentences/sec
Mean F1: 0.6235990126927694


## **중간 결과**

- 모델 : KoGPT2
- 특이사항
  - 질문과 관련된 키워드가 나오긴 하지만 답변이 좀 반복되고 난잡함
  - 외국어, 특수문자 등이 섞여서 나옴
  - 답변이 끝까지 안 나오고 짤리는 경우도 있음
  - Bert Score : 0.6280
    - 일반적으로 매우 낮거나 아쉬운 수준

- 개선 사항
  - 문장이 불필요하게 반복되고 잘리는 걸 보니 특수토큰이 빠진 건지 확인
  - 관련 키워드만 나오거나 난잡하게 생성되는 경우가 있어서 안정성 있게 생성되어야 할 것 같음
  - 반복되는 부분을 방지

#### 특수토큰 정의

In [32]:
def generation(input_text, model, isPrint=False):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(
        input_ids,
        max_length=250,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    
    if isPrint == True:
        print()
        print(output)
    return output

In [33]:
result_list = []

for i in range(3):
    print(f"TASK-{i}")
    result = compute_bertscore_from_sft_val_dataset(
        model=actor,
        val_dataset=bert_score_val_dataset,
        tokenizer=tokenizer,
        generation_fn=generation,
        lang="ko",
        num_samples=100,
        verbose=True,
    )

    result_list.append(result["mean_f1"])


# print("Mean Precision:", result["mean_precision"])
# print("Mean Recall:", result["mean_recall"])
print("Mean F1:", sum(result_list)/len(result_list))

TASK-0


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.



[Sample 0]
PROMPT:
### Instruction(명령어):
술 먹고 싶어

### Response(응답):
REFERENCE:
'술은 알코올이 함유된 음료수이며, 건강에 나쁜 영향을 미칠 수 있습니다. 따라서 술을 마시기 전에 본인의 건강 상태와 함께 적당한 양을 마시는 것이 중요합니다. 술을 마시기 전에 반드시 20세 이상인지 확인하시고, 규정된 양 이내에서 적절하게 즐기시길 바랍니다. 또한 술을 마시고 운전하는 것은 절대 금지하시기 바랍니다.
PREDICTION:
'좋은 술 드시겠어요! 어떤 음주나 다른 음료를 좋아하시나요? 음주 대신 조금 더 시원한 맥주나 리조트 메뉴를 즐기시면 좋을 것 같습니다.

[Sample 1]
PROMPT:
### Instruction(명령어):
쇼와 천황의 면책에 불만을 표한 프랑스의 판사는?

### Response(응답):
REFERENCE:
'제 2차 세계 대전 전후의 재앙적인 상황에서 쇼-고라제 정부와 천황을 비난하는 것은 위험하고 불안정한 시기였습니다. 따라서, 그런 판사는 적극적으로 찾아보기 어려웠으며, 불만을 표현하는 것 자체가 위험한 행동이었습니다. 그러나, 산더라이였라는 유명한 프랑스 판사는 쇼-고라제 정부와 천황을 비난하는 공개적인 선서를 한 인물로서 유명합니다. 이 선서는 파리 주재 미국 대사관에서 발표되었으며, 천황과 쇼-고라제 정부를 비판하고, 그들에 대한 책임을 요구하는 것으로 불리게 되었습니다. 산더라이 판사는 이 선서를 발표한 후 프랑스 정부에 의해 사형되었습니다.
PREDICTION:
'프랑스판사 보르텐예(Jorgen Jönene)입니다.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 0.56 seconds, 178.64 sentences/sec
Mean F1: 0.6982265114784241


#### temperature=0.75

In [34]:
def generation(input_text, model, isPrint=False):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(
        input_ids,
        max_length=250,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        temperature=0.75
    )
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    
    if isPrint == True:
        print()
        print(output)
    return output

In [35]:
result_list = []

for i in range(3):
    print(f"TASK-{i}")
    result = compute_bertscore_from_sft_val_dataset(
        model=actor,
        val_dataset=bert_score_val_dataset,
        tokenizer=tokenizer,
        generation_fn=generation,
        lang="ko",
        num_samples=100,
        verbose=True,
    )

    result_list.append(result["mean_f1"])


# print("Mean Precision:", result["mean_precision"])
# print("Mean Recall:", result["mean_recall"])
print("Mean F1:", sum(result_list)/len(result_list))

TASK-0

[Sample 0]
PROMPT:
### Instruction(명령어):
술 먹고 싶어

### Response(응답):
REFERENCE:
'술은 알코올이 함유된 음료수이며, 건강에 나쁜 영향을 미칠 수 있습니다. 따라서 술을 마시기 전에 본인의 건강 상태와 함께 적당한 양을 마시는 것이 중요합니다. 술을 마시기 전에 반드시 20세 이상인지 확인하시고, 규정된 양 이내에서 적절하게 즐기시길 바랍니다. 또한 술을 마시고 운전하는 것은 절대 금지하시기 바랍니다.
PREDICTION:
'안녕하세요! 술 마시는 것은 어떨까요?\n\n1. 커피를 마셔보고 싶으세요

[Sample 1]
PROMPT:
### Instruction(명령어):
쇼와 천황의 면책에 불만을 표한 프랑스의 판사는?

### Response(응답):
REFERENCE:
'제 2차 세계 대전 전후의 재앙적인 상황에서 쇼-고라제 정부와 천황을 비난하는 것은 위험하고 불안정한 시기였습니다. 따라서, 그런 판사는 적극적으로 찾아보기 어려웠으며, 불만을 표현하는 것 자체가 위험한 행동이었습니다. 그러나, 산더라이였라는 유명한 프랑스 판사는 쇼-고라제 정부와 천황을 비난하는 공개적인 선서를 한 인물로서 유명합니다. 이 선서는 파리 주재 미국 대사관에서 발표되었으며, 천황과 쇼-고라제 정부를 비판하고, 그들에 대한 책임을 요구하는 것으로 불리게 되었습니다. 산더라이 판사는 이 선서를 발표한 후 프랑스 정부에 의해 사형되었습니다.
PREDICTION:
'프랑스 판사는 판사의 임명권을 가지고 있지 않습니다. 프랑스 법원은 판사의 임명권을 갖는 것이 불가능하므로, 판사의 임명권을 가지고 있지 않습니다.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 0.45 seconds, 220.09 sentences/sec
Mean F1: 0.7067088087399801


In [56]:
torch.cuda.empty_cache()

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "EleutherAI/polyglot-ko-3.8b"

# tokenizer 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# model_max_length를 직접 제한
tokenizer.model_max_length = 512   # 필요하면 1024로 조절

# pad token이 없으면 eos token으로 대체
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# model 로드
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# pad_token_id도 맞춰두기
model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [15]:
# SFT_dataset 클래스를 사용해 훈련셋을 만들고 data collator 인스턴스를 만들기
train_dataset = SFTDataset(
    data_path='../KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl',
    tokenizer=tokenizer,
    split="train",
    eval_ratio=0.1,
    seed=42,
    verbose=True
)

val_dataset = SFTDataset(
    data_path='../KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl',
    tokenizer=tokenizer,
    split="eval",
    eval_ratio=0.1,
    seed=42,
    verbose=True
)

data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

bert_score_train_dataset = train_dataset
bert_score_val_dataset = val_dataset

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

Total samples: 12000
Train samples: 10800
Eval samples: 1200
Current split (train) samples: 10800


Total samples: 12000
Train samples: 10800
Eval samples: 1200
Current split (eval) samples: 1200


input : tensor([    6,     6,     6, 10276,  3864, 23184, 29828,    11,  7262,   348,
           12,    29,   202,  1198,  3648,   463,  5992,  8649,  2733,    34,
          202,   202,     6,     6,     6,  3069,  5862,    83,  2600,   426,
           11,  9643,    12,    29,    10,   828,   272,  5626, 29377,  3620,
          270,   453,    15,  1455, 10562,  3754,   274,  2708,  1610,   301,
         1384,   283,   327,   295,   536,   827,    17,  1146,  1754, 10562,
        23342,   463,  5992,   272,  8649,    15, 11765,    15, 12305,   433,
         1186,   288,  5335,   285,  5992,   301,  1203,  2136,    17,  1146,
         8649,   272,   924,   407,   333,  4213, 19665,   286,  5037,   327,
          316,   818,   274,    15,  8649,   301,  1203,   284,   272,   955,
          309,   750,   827,    17,   768,   488, 15337,  4548,  1793,  1641,
          274,  1026,  8129,   365,   327,  2556,  2403,   288,  1610,  5829,
          726,   274,  1912,   310,   458,  9061, 10710,

In [16]:
import transformers
from transformers import EarlyStoppingCallback

training_args = transformers.TrainingArguments(
    output_dir="test",
    num_train_epochs=10,                 # 충분히 크게 두고 early stopping에 맡기는 편이 일반적
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16=True,

    # evaluation / saving
    eval_strategy="epoch",               # 구버전에선 evaluation_strategy일 수 있음
    eval_steps=100,
    save_strategy="epoch",
    save_steps=100,
    save_total_limit=2,

    # best model / early stopping
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # 커스텀 collator 쓸 때 안전장치
    remove_unused_columns=False,
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # 또는 eval_dataset 변수명 사용
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [17]:
trainer.train()
model.save_pretrained('models/output_1_SFT_polyglot')

ValueError: Attempting to unscale FP16 gradients.